# Cat Breed Classification Using Convolutional Neural Networks (67 Breeds)

This notebook implements cat breed classification using deep learning techniques on the Kaggle Cat Breeds Dataset containing 67 different cat breeds. We use transfer learning with ResNet-50 and data augmentation for robust performance.

## Setup Instructions:

### For Kaggle:
- Upload this notebook to Kaggle
- The dataset will be automatically downloaded using kagglehub
- Run the notebook as-is

## Download the Dataset

Download the Kaggle Cat Breeds Dataset containing 67 different cat breeds.

In [ ]:
import kagglehub

# Download the Kaggle Cat Breeds Dataset
dataset_path = kagglehub.dataset_download("ma7555/cat-breeds-dataset")

print("Dataset downloaded to:", dataset_path)

# List the contents of the dataset
print("Dataset contents:")
for item in os.listdir(dataset_path):
    print(f"- {item}")

# Check if images are in a subfolder
images_path = os.path.join(dataset_path, "images")
if os.path.exists(images_path):
    print(f"\nImages found in subfolder: {images_path}")
    dataset_path = images_path  # Use the images subfolder as the dataset root
else:
    print("\nNo 'images' subfolder found, using root path")

## Install Required Packages

Install the necessary packages for this notebook. Note: On Kaggle, most packages are pre-installed.

In [ ]:
# Install required packages (uncomment if needed on Kaggle)
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# !pip install numpy pandas matplotlib seaborn scikit-learn tqdm pillow kagglehub

## Import Required Libraries

Import all necessary libraries for data processing, model building, and evaluation.

In [ ]:
import shutil
import os
import re
import copy
import random
import time
import sys
import glob
from PIL import Image

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn

import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

import sklearn
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder

import tqdm
from tqdm.notebook import tqdm as tqdm_notebook

# Set random seeds for reproducibility
def seed_torch(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_torch(42)

print('PyTorch version:', torch.__version__)
print('Torchvision version:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device count:', torch.cuda.device_count())
    print('Current CUDA device:', torch.cuda.current_device())
    print('CUDA device name:', torch.cuda.get_device_name())

## Explore the Dataset

Let's examine the structure of the downloaded dataset and understand what we're working with.

In [ ]:
# Explore the dataset structure
print(f"Dataset path: {dataset_path}")

# List all directories (breeds)
breed_dirs = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))]
print(f"\nNumber of breed folders: {len(breed_dirs)}")
print("\nFirst 10 breed folders:")
for i, breed in enumerate(sorted(breed_dirs)[:10]):
    print(f"{i+1}. {breed}")

# Count images per breed
breed_counts = {}
total_images = 0

for breed in breed_dirs:
    breed_path = os.path.join(dataset_path, breed)
    if os.path.exists(breed_path):
        images = [f for f in os.listdir(breed_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        count = len(images)
        breed_counts[breed] = count
        total_images += count

print(f"\nTotal images in folders: {total_images}")
print(f"Average images per breed: {total_images / len(breed_dirs):.1f}")

# Check if there's a CSV file with metadata
csv_path = os.path.join(dataset_path, "..", "data", "cats.csv")
if os.path.exists(csv_path):
    print(f"\n📄 Found CSV file: {csv_path}")
    cats_df = pd.read_csv(csv_path)
    print(f"CSV shape: {cats_df.shape}")
    print("CSV columns:", list(cats_df.columns))
    print("\nFirst 5 rows:")
    print(cats_df.head())
    
    # Check if CSV has breed information
    if 'breed' in cats_df.columns:
        print(f"\nBreed distribution in CSV (top 10):")
        print(cats_df['breed'].value_counts().head(10))
        
        # Compare with folder-based counts
        print(f"\n📊 Dataset Summary:")
        print(f"- CSV contains {len(cats_df)} cat records")
        print(f"- {len(cats_df['breed'].unique())} unique breeds in CSV")
        print(f"- {len(breed_dirs)} breed folders with images")
        print(f"- Total images in folders: {total_images}")
else:
    print("\nNo CSV file found in data folder")

## Validate and Filter Dataset

Check for corrupted images and filter them out to prevent training errors.


In [ ]:
# Create data loaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                         num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                       num_workers=0, pin_memory=True)

In [ ]:
def is_image_valid(image_path):
    """Check if an image file is valid and can be opened and converted"""
    try:
        with Image.open(image_path) as img:
            img.verify()  # Verify the image is not corrupted
            # Also try to load and convert to ensure it's fully readable
            img.load()
            img.convert("RGB")
        return True
    except (IOError, OSError, Image.DecompressionBombError, ValueError, TypeError):
        return False

# Validate all images in the dataset
print("Validating images in dataset...")
valid_samples = []

# Use the original dataset classes
temp_dataset = datasets.ImageFolder(root=dataset_path, transform=None)

for class_idx, class_name in enumerate(temp_dataset.classes):
    class_path = os.path.join(dataset_path, class_name)
    if os.path.isdir(class_path):
        image_files = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        valid_count = 0
        for img_file in image_files:
            img_path = os.path.join(class_path, img_file)
            if is_image_valid(img_path):
                valid_samples.append((img_path, class_idx))
                valid_count += 1
            else:
                print(f"Skipping corrupted image: {img_path}")
        print(f"Class '{class_name}': {valid_count}/{len(image_files)} valid images")

print(f"\nTotal valid images: {len(valid_samples)}")
print(f"Original dataset size: {len(temp_dataset)}")

# Create a new dataset with only valid images
from torch.utils.data import Dataset

class ValidImageDataset(Dataset):
    def __init__(self, samples, classes, transform=None):
        self.samples = samples
        self.transform = transform
        self.classes = classes

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label
        except Exception as e:
            # If image fails to load for any reason, return a placeholder
            print(f"Warning: Failed to load image {img_path}: {e}")
            if self.transform:
                # Create a placeholder image that matches the transform expectations
                placeholder = Image.new('RGB', (224, 224), color='black')
                return self.transform(placeholder), label
            else:
                placeholder = Image.new('RGB', (224, 224), color='black')
                return placeholder, label

# Replace the full_dataset with the filtered one
full_dataset = ValidImageDataset(valid_samples, temp_dataset.classes, transform=None)
print(f"Filtered dataset size: {len(full_dataset)}")

# Update class names and mappings
class_names = full_dataset.classes
class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}
idx_to_class = {v: k for k, v in class_to_idx.items()}

print(f"Number of classes: {len(class_names)}")
print("Classes:", class_names[:10], "..." if len(class_names) > 10 else "")

# Recreate train/validation split with filtered dataset
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size],
                                         generator=torch.Generator().manual_seed(42))

# Apply transforms
train_dataset.dataset.transform = train_transforms
val_dataset.dataset.transform = val_transforms

print(f"Updated training images: {len(train_dataset)}")
print(f"Updated validation images: {len(val_dataset)}")

# Recreate data loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                         num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                       num_workers=0, pin_memory=True)

print(f"Updated training batches: {len(train_loader)}")
print(f"Updated validation batches: {len(val_loader)}")

### Visualize Sample Images

Let's look at some sample images from different breeds to understand our data.

In [ ]:
# Visualize sample images
def imshow(inp, title=None):
    """Imshow for Tensor."""
    inp = inp.numpy().transpose((1, 2, 0))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    inp = std * inp + mean
    inp = np.clip(inp, 0, 1)
    plt.imshow(inp)
    if title is not None:
        plt.title(title)
    plt.pause(0.001)  # pause a bit so that plots are updated

# Get a batch of training data
inputs, classes = next(iter(train_loader))

# Make a grid from batch
out = torchvision.utils.make_grid(inputs[:8])  # Show first 8 images

fig = plt.figure(figsize=(12, 6))
imshow(out)
plt.title('Sample Training Images')
plt.axis('off')
plt.show()

# Show class names for the batch
print("Classes in this batch:")
for i in range(min(8, len(classes))):
    print(f"Image {i+1}: {idx_to_class[classes[i].item()]}")

# Show distribution of classes in one batch
batch_classes = [idx_to_class[c.item()] for c in classes]
unique, counts = np.unique(batch_classes, return_counts=True)
plt.figure(figsize=(10, 5))
plt.bar(unique, counts)
plt.xticks(rotation=45, ha='right')
plt.title('Class Distribution in One Batch')
plt.ylabel('Count')
plt.show()

## Build the Model

We'll use ResNet-50 with transfer learning, adapting it for our 67-class classification task.

In [ ]:
# Load pretrained ResNet-50 model
model = torchvision.models.resnet50(pretrained=True)

# Freeze all layers initially
for param in model.parameters():
    param.requires_grad = False

# Get the number of input features for the final fully connected layer
num_ftrs = model.fc.in_features
print(f"ResNet-50 fc input features: {num_ftrs}")

# Replace the final layer for our 67-class classification
num_classes = len(class_names)
model.fc = nn.Sequential(
    nn.Linear(num_ftrs, 512),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(512, num_classes),
    nn.LogSoftmax(dim=1)
)

# Move model to device
model = model.to(device)

print(f"Model created with {num_classes} output classes")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Loss function and optimizer
criterion = nn.NLLLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# Learning rate scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

print("Model, loss function, optimizer, and scheduler ready!")

## Training Utilities

Define functions for training, validation, and evaluation.

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train the model for one epoch"""
    model.train()
    running_loss = 0.0
    running_corrects = 0

    for inputs, labels in tqdm_notebook(train_loader, desc='Training', leave=False):
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = running_corrects.double() / len(train_loader.dataset)

    return epoch_loss, epoch_acc.item()

def validate_epoch(model, val_loader, criterion, device):
    """Validate the model for one epoch"""
    model.eval()
    running_loss = 0.0
    running_corrects = 0

    with torch.no_grad():
        for inputs, labels in tqdm_notebook(val_loader, desc='Validating', leave=False):
            inputs = inputs.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)

            # Statistics
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

    epoch_loss = running_loss / len(val_loader.dataset)
    epoch_acc = running_corrects.double() / len(val_loader.dataset)

    return epoch_loss, epoch_acc.item()

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                num_epochs=25, device='cuda', save_path='best_model.pth'):
    """Train the model and return training history"""

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': []
    }

    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        # Train
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        print(f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f}')

        # Validate
        val_loss, val_acc = validate_epoch(model, val_loader, criterion, device)
        print(f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}')

        # Update learning rate
        scheduler.step()

        # Save history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        # Deep copy the model if it's the best accuracy
        if val_acc > best_acc:
            best_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), save_path)
            print(f'Best model saved with accuracy: {best_acc:.4f}')

    print(f'\nBest validation accuracy: {best_acc:.4f}')

    # Load best model weights
    model.load_state_dict(best_model_wts)
    return model, history

## Train the Model

Train the ResNet-50 model with transfer learning on our cat breed dataset.

In [ ]:
# Train the model
num_epochs = 15  # Adjust based on your needs and time constraints
save_path = 'cat_breed_classifier.pth'

# Ensure we're using the filtered dataset
if not hasattr(train_loader.dataset.dataset, 'samples'):
    print("Error: Dataset not properly filtered. Please run the validation cell first.")
else:
    print("Starting training...")
    start_time = time.time()

    trained_model, history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        num_epochs=num_epochs,
        device=device,
        save_path=save_path
    )

    training_time = time.time() - start_time
    print(f"Training completed in {training_time:.2f} seconds")

    # Plot training history
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Training Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Training and Validation Loss')

    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Training Accuracy')
    plt.plot(history['val_acc'], label='Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.title('Training and Validation Accuracy')

    plt.tight_layout()
    plt.show()

    print(f"Final training accuracy: {history['train_acc'][-1]:.4f}")
    print(f"Final validation accuracy: {history['val_acc'][-1]:.4f}")

## Evaluate the Model

Evaluate the trained model on the validation set and generate detailed metrics.

In [ ]:
def evaluate_model(model, data_loader, device, class_names):
    """Evaluate model and return predictions, true labels, and metrics"""
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for inputs, labels in tqdm_notebook(data_loader, desc='Evaluating'):
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            probs = torch.exp(outputs)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    return np.array(all_preds), np.array(all_labels), np.array(all_probs)

# Evaluate on validation set
print("Evaluating model on validation set...")
val_preds, val_labels, val_probs = evaluate_model(trained_model, val_loader, device, class_names)

# Calculate accuracy
val_accuracy = np.mean(val_preds == val_labels)
print(f"Validation Accuracy: {val_accuracy:.4f}")

# Classification report
from sklearn.metrics import classification_report
print("\nClassification Report:")
print(classification_report(val_labels, val_preds, target_names=class_names,
                          labels=range(len(class_names)), digits=4))

# Confusion matrix
cm = confusion_matrix(val_labels, val_preds)
plt.figure(figsize=(20, 16))
sn.heatmap(cm, annot=False, cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Top N accuracy
def top_n_accuracy(probs, labels, n=3):
    """Calculate top-N accuracy"""
    top_n_preds = np.argsort(probs, axis=1)[:, -n:]
    hits = 0
    for i, label in enumerate(labels):
        if label in top_n_preds[i]:
            hits += 1
    return hits / len(labels)

top1_acc = np.mean(val_preds == val_labels)
top3_acc = top_n_accuracy(val_probs, val_labels, n=3)
top5_acc = top_n_accuracy(val_probs, val_labels, n=5)

print(f"\nAccuracy Metrics:")
print(f"Top-1 Accuracy: {top1_acc:.4f}")
print(f"Top-3 Accuracy: {top3_acc:.4f}")
print(f"Top-5 Accuracy: {top5_acc:.4f}")

# Per-class accuracy
class_correct = np.zeros(len(class_names))
class_total = np.zeros(len(class_names))

for pred, label in zip(val_preds, val_labels):
    class_correct[label] += (pred == label)
    class_total[label] += 1

class_accuracy = class_correct / class_total

# Show worst and best performing classes
class_acc_df = pd.DataFrame({
    'Breed': class_names,
    'Accuracy': class_accuracy,
    'Total_Samples': class_total.astype(int)
})

class_acc_df = class_acc_df.sort_values('Accuracy', ascending=False)

print(f"\nTop 10 Best Performing Breeds:")
print(class_acc_df.head(10).to_string(index=False))

print(f"\nTop 10 Worst Performing Breeds:")
print(class_acc_df.tail(10).to_string(index=False))

## Visualize Predictions

Let's see how our model performs on some sample images.

In [ ]:
# Get some random validation images to test
val_dataset_full = full_dataset  # Use the filtered dataset
val_dataset_full.transform = val_transforms  # Apply validation transforms
val_indices = np.random.choice(len(val_dataset), 6, replace=False)

print("Sample Predictions:")
for idx in val_indices:
    img_path, label = val_dataset.dataset.samples[val_dataset.indices[idx]]
    predict_and_show(trained_model, img_path, val_transforms, device, class_names)

## Save Model and Summary

Save the trained model and provide a summary of the results.

In [ ]:
# Save the final model
final_model_path = 'cat_breed_classifier_final.pth'
torch.save({
    'model_state_dict': trained_model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'class_names': class_names,
    'num_classes': num_classes,
    'history': history,
    'val_accuracy': val_accuracy,
    'top3_accuracy': top3_acc,
    'top5_accuracy': top5_acc
}, final_model_path)

print(f"Model saved to: {final_model_path}")

# Create a summary
print("\n" + "="*60)
print("CAT BREED CLASSIFICATION MODEL SUMMARY")
print("="*60)
print(f"Dataset: Kaggle Cat Breeds Dataset")
print(f"Number of classes: {num_classes}")
print(f"Total images: {len(full_dataset)}")
print(f"Training images: {len(train_dataset)}")
print(f"Validation images: {len(val_dataset)}")
print()
print(f"Model: ResNet-50 with transfer learning")
print(f"Training epochs: {num_epochs}")
print(f"Batch size: {batch_size}")
print(f"Training time: {training_time:.2f} seconds")
print()
print("PERFORMANCE METRICS:")
print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Top-3 Accuracy: {top3_acc:.4f}")
print(f"Top-5 Accuracy: {top5_acc:.4f}")
print()
print("TRAINING HISTORY:")
print(f"Initial training loss: {history['train_loss'][0]:.4f}")
print(f"Final training loss: {history['train_loss'][-1]:.4f}")
print(f"Initial validation loss: {history['val_loss'][0]:.4f}")
print(f"Final validation loss: {history['val_loss'][-1]:.4f}")
print(f"Initial training accuracy: {history['train_acc'][0]:.4f}")
print(f"Final training accuracy: {history['train_acc'][-1]:.4f}")
print(f"Initial validation accuracy: {history['val_acc'][0]:.4f}")
print(f"Final validation accuracy: {history['val_acc'][-1]:.4f}")
print()
print("DATA AUGMENTATION USED:")
print("- Random horizontal flip")
print("- Random rotation (±15°)")
print("- Color jitter (brightness, contrast, saturation, hue)")
print("- Random crop and resize")
print()
print("MODEL ARCHITECTURE:")
print("- Pretrained ResNet-50 backbone (frozen)")
print("- Custom classifier head:")
print("  - Linear(2048 → 512)")
print("  - ReLU activation")
print("  - Dropout(0.3)")
print("  - Linear(512 → 67)")
print("  - LogSoftmax")
print("="*60)

# Save class names for future use
import json
with open('class_names.json', 'w') as f:
    json.dump(class_names, f)

print("Class names saved to: class_names.json")
print("\nNotebook completed! You can now use this model to classify cat breeds.")

## Manual Image Testing

Test the trained model on your own cat images. You can upload images to your Kaggle notebook and test the model's predictions.

In [ ]:
def test_random_validation_images(num_images=3):
    print(f"Testing {num_images} random images from validation set...")

    # Get random indices from validation set
    val_indices = np.random.choice(len(val_dataset), min(num_images, len(val_dataset)), replace=False)

    for i, idx in enumerate(val_indices):
        # Get image path and true label
        img_path, true_label = val_dataset.dataset.samples[val_dataset.indices[idx]]
        true_breed = class_names[true_label]

        print(f"\n{'='*60}")
        print(f"🖼️  TEST IMAGE {i+1}: True breed = {true_breed}")
        print(f"{'='*60}")

        # Make prediction
        result = predict_single_image(trained_model, img_path, test_transform, device, class_names)
        if result:
            image, top_probs, top_classes = result
            display_prediction_results(image, top_probs, top_classes, class_names, f"Validation Image {i+1}")

            # Check if prediction is correct
            predicted_breed = class_names[top_classes[0]]
            is_correct = predicted_breed == true_breed
            status = "✅ CORRECT!" if is_correct else "❌ INCORRECT"
            print(f"Result: {status} (Predicted: {predicted_breed})")
        else:
            print("❌ Failed to process image.")